In [1]:
import os
import sys

# ensure src is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

PROJECT_ROOT: /Users/seleneandrade/Documents/helu/sele/data-challenge/src


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[1]")
    .appName("GoldPrototype")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark: {spark.version}")

26/03/14 13:43:49 WARN Utils: Your hostname, MacBook-Pro-de-Selene.local resolves to a loopback address: 127.0.0.1; using 192.168.1.146 instead (on interface en0)
26/03/14 13:43:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/opt/homebrew/Caskroom/miniconda/base/envs/helu/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/seleneandrade/.ivy2/cache
The jars for the packages stored in: /Users/seleneandrade/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6dcd6214-0c34-4b52-93a1-995b5a622157;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 335ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default  

Spark: 3.5.6


In [5]:
from src.helu.utils.config import SILVER_DIR

exchange_df = spark.read.format("delta").load(f"{SILVER_DIR}/exchange_rates")
apfel_df = spark.read.format("delta").load(f"{SILVER_DIR}/apfel")
fenster_df = spark.read.format("delta").load(f"{SILVER_DIR}/fenster")

In [8]:
exchange_df.toPandas()

,date,currency,rate_to_eur,_ingestion_timestamp_utc,_source,_inserted_date_utc,_silver_ingestion_timestamp_utc
0,2025-02-01,USD,0.92,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046
1,2025-03-01,USD,0.91,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046
2,2025-04-01,USD,0.90,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046
3,2025-05-01,USD,0.89,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046
4,2025-06-01,USD,0.88,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046
5,2025-07-01,USD,0.87,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046
6,2025-08-01,USD,0.86,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046
7,2025-09-01,USD,0.85,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046
8,2025-10-01,USD,0.84,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046
9,2025-11-01,USD,0.83,2026-03-14 13:41:04.908348,exchange_rates,2026-03-14,2026-03-14 13:41:17.273046


In [140]:
print(exchange_df.count())  # 13
print(apfel_df.count())  # 757
print(fenster_df.count())  # 547

13


757


547


In [81]:
exchange_df.createOrReplaceTempView("exchange_rates")
apfel_df.createOrReplaceTempView("apfel")
fenster_df.createOrReplaceTempView("fenster")

In [69]:
print("Apfel values:")
# spark.sql("SELECT event_timestamp, country_code, subscription_plan, renewal_period, currency_norm, price_amount, _source, subscription_status, _silver_ingestion_timestamp_utc FROM apfel").show()

spark.sql("SELECT country_code FROM apfel group by  country_code").show()
spark.sql("SELECT renewal_period FROM apfel group by  renewal_period").show()
spark.sql("SELECT currency_norm FROM apfel group by  currency_norm").show()
spark.sql("SELECT subscription_plan FROM apfel group by  subscription_plan").show()

Apfel values:


+------------+
|country_code|
+------------+
|          DE|
|          GB|
+------------+



+--------------+
|renewal_period|
+--------------+
|        yearly|
|       monthly|
+--------------+



+-------------+
|currency_norm|
+-------------+
|          GBP|
|          EUR|
|          USD|
+-------------+



+-----------------+
|subscription_plan|
+-----------------+
|          premium|
|         standard|
+-----------------+



In [82]:
print("Fenster values:")
# spark.sql("""SELECT event_timestamp,
#                 subscription_status,
#                 country_norm,
#                 currency,
#                 price_amount,
#                 _source,
#                 renewal_period,
#                 subscription_plan, _silver_ingestion_timestamp_utc
#              FROM fenster""").show()

spark.sql(
    "SELECT subscription_status FROM fenster group by  subscription_status"
).show()
spark.sql("SELECT country_norm FROM fenster group by  country_norm").show()
spark.sql("SELECT currency FROM fenster group by  currency").show()
spark.sql("SELECT renewal_period FROM fenster group by  renewal_period").show()
spark.sql("SELECT subscription_plan FROM fenster group by  subscription_plan").show()

Fenster values:


+-------------------+
|subscription_status|
+-------------------+
|             cancel|
|              renew|
|                new|
+-------------------+



+------------+
|country_norm|
+------------+
|         USA|
|          GB|
+------------+



+--------+
|currency|
+--------+
|     USD|
+--------+



+--------------+
|renewal_period|
+--------------+
|        yearly|
|       monthly|
+--------------+

+-----------------+
|subscription_plan|
+-----------------+
|          premium|
|         standard|
+-----------------+



In [114]:
apfer_eur_query = """
select
    _source as platform,
    subscription_plan as subscription_type,
    country_code as country,
    DATE_FORMAT(event_timestamp, 'yyyy-MM') as report_month,
    CASE subscription_status
        WHEN 'started'   THEN 'acquisition'
        WHEN 'renewed'   THEN 'renewal'
        WHEN 'cancelled' THEN 'cancellation'
        ELSE subscription_status
    END as subscription_status,
    price_amount,
    renewal_period,
    1.0 as rate_to_eur,
    currency_norm as original_currency
    from apfel
    where currency_norm = 'EUR'

"""

spark.sql(apfer_eur_query).toPandas()

apfel_eur_df = spark.sql(apfer_eur_query)

In [115]:
apfel_not_eur_query = """
select
    a._source as platform,
    subscription_plan as subscription_type,
    country_code as country,
    DATE_FORMAT(event_timestamp, 'yyyy-MM') as report_month,
    CASE subscription_status
        WHEN 'started'   THEN 'acquisition'
        WHEN 'renewed'   THEN 'renewal'
        WHEN 'cancelled' THEN 'cancellation'
        ELSE subscription_status
    END as subscription_status,
    price_amount,
    renewal_period,
    COALESCE(e.rate_to_eur, 1.0) as rate_to_eur,
    a.currency_norm as original_currency
    from apfel a
    left join exchange_rates e
        on  a.currency_norm = e.currency
        and DATE_TRUNC('month', a.event_timestamp) = e.date
    where currency_norm != 'EUR'
"""

spark.sql(apfel_not_eur_query).toPandas()
apfel_not_eur_df = spark.sql(apfel_not_eur_query)

In [124]:
fenster_query = """
    select f._source as platform,
    subscription_plan as subscription_type,
    country_norm as country,
    DATE_FORMAT(event_timestamp, 'yyyy-MM') as report_month,
    CASE subscription_status
        WHEN 'new'   THEN 'acquisition'
        WHEN 'renew'   THEN 'renewal'
        WHEN 'cancel' THEN 'cancellation'
        ELSE subscription_status
    END as subscription_status,
    price_amount,
    renewal_period,
    COALESCE(e.rate_to_eur, 1.0) as rate_to_eur,
    f.currency as original_currency
    from fenster f
    left join exchange_rates e
    on  f.currency = e.currency
    and DATE_TRUNC('month', f.event_timestamp) = e.date
"""

spark.sql(fenster_query).toPandas()
fenster_df = spark.sql(fenster_query)

In [125]:
apfel_eur_df.createOrReplaceTempView("apfel_eur")
apfel_not_eur_df.createOrReplaceTempView("apfel_not_eur")
fenster_df.createOrReplaceTempView("fenster_norm")

In [126]:
combined_query = """
select * from apfel_eur
union all
select * from apfel_not_eur
union all
select * from fenster_norm
"""

combined_df = spark.sql(combined_query)
combined_df.createOrReplaceTempView("combined_df")

combined_df.toPandas()

,platform,subscription_type,country,report_month,subscription_status,price_amount,renewal_period,rate_to_eur,original_currency
0,apfel,premium,DE,2025-12,renewal,119.99,yearly,1.00,EUR
1,apfel,standard,DE,2025-04,cancellation,59.99,yearly,1.00,EUR
2,apfel,standard,GB,2025-02,acquisition,59.99,yearly,1.00,EUR
3,apfel,premium,GB,2025-03,cancellation,12.99,monthly,1.00,EUR
4,apfel,premium,GB,2025-02,acquisition,12.99,monthly,1.00,EUR
...,...,...,...,...,...,...,...,...,...
1299,fenster,standard,GB,2026-02,acquisition,69.99,yearly,0.84,USD
1300,fenster,standard,USA,2026-02,acquisition,69.99,yearly,0.84,USD
1301,fenster,standard,USA,2026-02,acquisition,6.99,monthly,0.84,USD
1302,fenster,standard,GB,2026-02,acquisition,69.99,yearly,0.84,USD


Your pipeline should evaluate and fix any data quality issues you identify.

The consolidated report must include at least the following attributes: platform, subscription_type, country, acquisitions, renewals, cancellations, mrr_eur.

Subscription events and exchange rates should be ingested from the Docker-based API (see Data Sources below).

The report should be stored in a queryable format (please document reasons for your storage decision).
The pipeline should be idempotent.

In [110]:
print("combined values:")

# spark.sql("select platform, count(platform) from combined_df group by platform").show()
# spark.sql("select subscription_type, count(subscription_type) from combined_df group by subscription_type").show()
# spark.sql("select country, count(country) from combined_df group by country").show()
spark.sql(
    "select platform,subscription_type, country,subscription_status, count(*) from combined_df group by platform,subscription_type, country, subscription_status"
).show()

combined values:


+--------+-----------------+-------+-------------------+--------+
|platform|subscription_type|country|subscription_status|count(1)|
+--------+-----------------+-------+-------------------+--------+
|   apfel|          premium|     GB|       cancellation|      17|
|   apfel|          premium|     DE|        acquisition|      82|
|   apfel|          premium|     DE|       cancellation|      22|
|   apfel|          premium|     GB|            renewal|      72|
|   apfel|         standard|     DE|            renewal|      79|
|   apfel|         standard|     GB|       cancellation|      21|
|   apfel|         standard|     GB|            renewal|      64|
|   apfel|          premium|     DE|            renewal|      74|
|   apfel|         standard|     DE|       cancellation|      32|
|   apfel|         standard|     GB|        acquisition|     101|
|   apfel|          premium|     GB|        acquisition|      98|
|   apfel|         standard|     DE|        acquisition|      95|
| fenster|

In [127]:
mrr_calculated_query = """
select
    platform,
    subscription_type,
    country,
    report_month,
    case when renewal_period = 'monthly' then price_amount
         when renewal_period = 'yearly' then price_amount / 12
         else price_amount end as original_monthly_amount,

    case when renewal_period = 'monthly' then round(price_amount * rate_to_eur, 2)
         when renewal_period = 'yearly' then round((price_amount / 12) * rate_to_eur, 2)
         else price_amount end as eur_monthly_amount
    from combined_df
    where subscription_status in ('acquisition', 'renewal')
"""

mrr_calculated_df = spark.sql(mrr_calculated_query)
mrr_calculated_df.createOrReplaceTempView("mrr_calculated_df")

mrr_calculated_df.toPandas()

,platform,subscription_type,country,report_month,original_monthly_amount,eur_monthly_amount
0,apfel,premium,DE,2025-12,9.999167,10.00
1,apfel,standard,GB,2025-02,4.999167,5.00
2,apfel,premium,GB,2025-02,12.990000,12.99
3,apfel,premium,DE,2025-02,9.999167,10.00
4,apfel,premium,DE,2026-01,9.999167,10.00
...,...,...,...,...,...,...
1146,fenster,standard,GB,2026-02,5.832500,4.90
1147,fenster,standard,USA,2026-02,5.832500,4.90
1148,fenster,standard,USA,2026-02,6.990000,5.87
1149,fenster,standard,GB,2026-02,5.832500,4.90


In [131]:
final_report_query = """
select
    c.platform,
    c.subscription_type,
    c.country,
    c.report_month,
    c.original_currency,
    count(CASE WHEN c.subscription_status = 'acquisition'  THEN 1 END) as acquisitions,
    count(CASE WHEN c.subscription_status = 'renewal'      THEN 1 END) as renewals,
    count(CASE WHEN c.subscription_status = 'cancellation' THEN 1 END) as cancellations,
    COALESCE(SUM(m.eur_monthly_amount), 0) as mrr_eur
from combined_df c
    left join mrr_calculated_df m
        on  c.platform          = m.platform
        and c.subscription_type = m.subscription_type
        and c.country           = m.country
        and c.report_month      = m.report_month
    group by
        c.platform,
        c.subscription_type,
        c.country,
        c.original_currency,
        c.report_month
    order by
        c.report_month,
        c.platform,
        c.country,
        c.subscription_type
"""
spark.sql(final_report_query).toPandas()

,platform,subscription_type,country,report_month,original_currency,acquisitions,renewals,cancellations,mrr_eur
0,apfel,premium,DE,2025-02,EUR,2,2,2,60.00
1,apfel,standard,DE,2025-02,EUR,1,0,2,17.97
2,apfel,premium,GB,2025-02,EUR,16,0,0,183.92
3,apfel,standard,GB,2025-02,EUR,6,3,3,63.96
4,fenster,standard,GB,2025-02,USD,1,0,0,5.37
...,...,...,...,...,...,...,...,...,...
101,fenster,premium,GB,2026-02,USD,104,65,0,2073.37
102,fenster,standard,GB,2026-02,USD,28,21,0,260.47
103,fenster,premium,USA,2026-02,USD,120,24,0,1687.56
104,fenster,standard,USA,2026-02,USD,30,6,12,242.96


In [3]:
from src.helu.utils.config import GOLD_DIR

final_df = spark.read.format("delta").load(f"{GOLD_DIR}/financial_report")

In [4]:
final_df.count()  # 106

26/03/14 13:44:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


106

In [143]:
final_df.toPandas()

,platform,subscription_type,country,report_month,original_currency,acquisitions,renewals,cancellations,mrr_eur,_gold_ingestion_timestamp_utc,_inserted_date_utc
0,fenster,premium,GB,2025-03,USD,72,72,108,3151.26,2026-03-13 23:08:25.936775,2026-03-13
1,fenster,standard,GB,2025-03,USD,216,108,108,2596.32,2026-03-13 23:08:25.936775,2026-03-13
2,fenster,premium,USA,2025-03,USD,36,0,18,736.56,2026-03-13 23:08:25.936775,2026-03-13
3,fenster,standard,USA,2025-03,USD,36,0,0,228.96,2026-03-13 23:08:25.936775,2026-03-13
4,apfel,premium,DE,2025-04,EUR,1,0,2,30.00,2026-03-13 23:08:25.936775,2026-03-13
...,...,...,...,...,...,...,...,...,...,...,...
101,fenster,premium,GB,2025-05,USD,54,27,54,1701.00,2026-03-13 23:08:25.936775,2026-03-13
102,fenster,standard,GB,2025-05,USD,432,144,72,3696.84,2026-03-13 23:08:25.936775,2026-03-13
103,fenster,premium,USA,2025-05,USD,144,0,0,1841.04,2026-03-13 23:08:25.936775,2026-03-13
104,fenster,standard,USA,2025-05,USD,360,216,144,4200.30,2026-03-13 23:08:25.936775,2026-03-13


In [133]:
final_df.toPandas()

,platform,subscription_type,country,report_month,original_currency,acquisitions,renewals,cancellations,mrr_eur,_gold_ingestion_timestamp_utc,_inserted_date_utc
0,fenster,premium,GB,2025-03,USD,8,8,12,350.14,2026-03-13 22:40:30.263873,2026-03-13
1,fenster,standard,GB,2025-03,USD,24,12,12,288.48,2026-03-13 22:40:30.263873,2026-03-13
2,fenster,premium,USA,2025-03,USD,4,0,2,81.84,2026-03-13 22:40:30.263873,2026-03-13
3,fenster,standard,USA,2025-03,USD,4,0,0,25.44,2026-03-13 22:40:30.263873,2026-03-13
4,apfel,premium,DE,2025-04,EUR,1,0,2,30.00,2026-03-13 22:40:30.263873,2026-03-13
...,...,...,...,...,...,...,...,...,...,...,...
101,fenster,premium,GB,2025-05,USD,6,3,6,189.00,2026-03-13 22:40:30.263873,2026-03-13
102,fenster,standard,GB,2025-05,USD,48,16,8,410.76,2026-03-13 22:40:30.263873,2026-03-13
103,fenster,premium,USA,2025-05,USD,16,0,0,204.56,2026-03-13 22:40:30.263873,2026-03-13
104,fenster,standard,USA,2025-05,USD,40,24,16,466.70,2026-03-13 22:40:30.263873,2026-03-13
